# Esame pratico Modulo 2 — Data Engineering Pipeline

Progetto finale basato sul dataset generato localmente tramite `generator.py`.

## Obiettivi
- Ingestion di file multipli con Pandas e Dask
- Pipeline ETL con PySpark
- Reporting con Matplotlib / Seaborn
- Bonus: streaming con PySpark Structured Streaming

## Nota sul dataset
La traccia richiede un'aggregazione per `payment_type`, ma il dataset generato da `generator.py` non contiene questa colonna.
Le transazioni disponibili includono invece: `transaction_id`, `customer_id`, `product_id`, `region_id`, `quantity`, `amount`, `ts`, `year`, `month`.

Per l'esercizio Dask, l'aggregazione verrà quindi eseguita su una colonna realmente presente nel dataset: `region_id`

In [23]:
from pathlib import Path
import glob
import os

import pandas as pd
import dask.dataframe as dd

BASE_DIR = Path("data_local")
PARQUET_DIR = BASE_DIR / "parquet"
JSON_DIR = BASE_DIR / "json"

print("Cartella base:", BASE_DIR.resolve())
print("File parquet:", len(list(PARQUET_DIR.glob("*.parquet"))))
print("File jsonl:", len(list(JSON_DIR.glob("*.jsonl"))))

Cartella base: /Users/francesco/Workspace/Epicode_Python_AI_ML/repos/first-project/Modulo_2_Python_Data_science/PySPARK/modulo2_esame/data_local
File parquet: 8
File jsonl: 5


In [24]:
parquet_files = sorted(PARQUET_DIR.glob("*.parquet"))
json_files = sorted(JSON_DIR.glob("*.jsonl"))

print("Primi file parquet:")
for file in parquet_files[:5]:
    print("-", file.name)

print("\nPrimi file jsonl:")
for file in json_files[:5]:
    print("-", file.name)

Primi file parquet:
- customers.parquet
- products.parquet
- regions.parquet
- transactions_batch_0000.parquet
- transactions_batch_0001.parquet

Primi file jsonl:
- transactions_part_0000.jsonl
- transactions_part_0001.jsonl
- transactions_part_0002.jsonl
- transactions_part_0003.jsonl
- transactions_part_0004.jsonl


In [25]:
sample_json = pd.read_json(json_files[0], lines=True)

print("Shape file campione:", sample_json.shape)
print("\nColonne:")
print(sample_json.columns.tolist())

sample_json.head()

Shape file campione: (100000, 9)

Colonne:
['transaction_id', 'customer_id', 'product_id', 'region_id', 'quantity', 'amount', 'ts', 'year', 'month']


,transaction_id,customer_id,product_id,region_id,quantity,amount,ts,year,month
0,5bc9c52f-4c97-42ad-ab30-9883399a18ac,16676,2337,2,1,60.130001,2020-08-31,2020,8
1,133aae84-94f7-43e5-b697-c5aedbb4a362,33111,3963,1,1,18.030001,2020-12-06,2020,12
2,a0f07d81-6eef-44d1-a419-58f691434830,9831,175,1,4,47.919998,2022-04-04,2022,4
3,96037256-4891-414b-bdc6-bd90d1e848eb,28516,4459,1,4,229.600006,2020-09-30,2020,9
4,a2a09db4-d40e-49e1-9b5b-50c5182727b3,2362,3713,4,5,44.650002,2021-03-31,2021,3


In [26]:
sample_json.dtypes

transaction_id     object
customer_id         int64
product_id          int64
region_id           int64
quantity            int64
amount            float64
ts                 object
year                int64
month               int64
dtype: object

## Esercizio 1A — Ingestion con Pandas
Lettura sequenziale dei file JSONL per contenere l'uso della memoria e calcolo del totale generale della colonna `amount`.

In [27]:
def somma_amount_con_pandas(file_list):
    """
    Legge i file JSONL uno alla volta con Pandas, calcola la somma
    della colonna 'amount' per ciascun file e restituisce il totale generale.

    Parameters
    ----------
    file_list : list[Path]
        Lista dei file JSONL da elaborare.

    Returns
    -------
    float
        Somma totale della colonna 'amount' su tutti i file.
    """
    totale_generale = 0.0

    for file_path in file_list:
        df = pd.read_json(file_path, lines=True)
        totale_file = df["amount"].sum()
        totale_generale += totale_file

        print(f"{file_path.name}: totale amount = {totale_file:.2f}")

    return totale_generale

In [28]:
totale_amount_pandas = somma_amount_con_pandas(json_files)

print("\nTotale generale amount (Pandas):", round(totale_amount_pandas, 2))

transactions_part_0000.jsonl: totale amount = 9815541.99
transactions_part_0001.jsonl: totale amount = 9783011.98
transactions_part_0002.jsonl: totale amount = 9820215.26
transactions_part_0003.jsonl: totale amount = 9815461.27
transactions_part_0004.jsonl: totale amount = 9794082.15

Totale generale amount (Pandas): 49028312.65


## Esercizio 1B — Ingestion con Dask
Lettura di tutti i file JSONL con wildcard e calcolo della media degli importi raggruppati per `region_id`.

In [29]:
dask_df = dd.read_json(str(JSON_DIR / "*.jsonl"), lines=True)

print("Colonne Dask:")
print(dask_df.columns.tolist())

Colonne Dask:
['transaction_id', 'customer_id', 'product_id', 'region_id', 'quantity', 'amount', 'ts', 'year', 'month']


In [30]:
media_amount_per_regione = (
    dask_df.groupby("region_id")["amount"]
    .mean()
    .compute()
    .sort_index()
)

print("Media amount per region_id:")
print(media_amount_per_regione)

Media amount per region_id:
region_id
1    98.338746
2    98.103429
3    97.441572
4    98.406569
5    97.997947
Name: amount, dtype: float64


In [31]:
print("Totale file JSONL letti:", len(json_files))
print("Numero regioni aggregate con Dask:", media_amount_per_regione.shape[0])

Totale file JSONL letti: 5
Numero regioni aggregate con Dask: 5
